# Python UDFs & Control Flow

The previous notebook covered SQL UDFs. This notebook covers **Python UDFs** — when you need the full power of Python (libraries, complex logic, ML model inference) inside a Spark transformation — and **notebook control flow** — how Databricks notebooks can be parameterized, chained, and orchestrated.

In this notebook we'll cover:
- Python UDFs — creation, SQL registration, and the performance cost
- Pandas UDFs (vectorized) — dramatically faster than row-at-a-time Python UDFs
- `dbutils.widgets` — parameterizing notebooks for job execution
- `dbutils.notebook.run` / `exit` — chaining notebooks
- Mixing Python variables with `spark.sql()` calls

## Python UDFs

### Creating a Python UDF

Two equivalent patterns — decorator and explicit wrapper:

```python
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Pattern 1 — decorator
@udf(returnType=StringType())
def mask_email(email: str) -> str:
    if email is None:
        return None
    local, domain = email.split('@')
    return local[:2] + '*' * (len(local) - 2) + '@' + domain

# Pattern 2 — explicit wrap
def _mask_email(email):
    ...

mask_email = udf(_mask_email, StringType())
```

### Using a Python UDF in PySpark

```python
df.withColumn("masked", mask_email(col("email")))
```

### Registering for SQL

To use a Python UDF in a `spark.sql()` string, register it with a name:

```python
spark.udf.register("mask_email", mask_email, StringType())

# Now callable from SQL
spark.sql("SELECT mask_email(email) FROM customers")
```

> Registering via `spark.udf.register` creates a **session-scoped** (temporary) function — it is not persisted in the metastore. Every session that needs it must re-register it.

## The Python UDF Performance Problem

Every Python UDF call crosses the JVM-Python boundary **once per row**:

```
JVM (Spark executor)  ──serialize──▶  Python process  ──deserialize──▶  JVM
     row data            (pickle)       UDF executes        result
```

For a 100M-row table, that's 100M serialization round-trips. This is the primary reason Python UDFs are slow.

| | Built-in / SQL UDF | Python UDF | Pandas UDF |
|---|---|---|---|
| Runs on | JVM | Python (row-at-a-time) | Python (batch via Arrow) |
| Serialization | None | Per row (pickle) | Per batch (Arrow — fast) |
| Catalyst optimization | Full / None | None | None |
| Use when | Always first choice | No SQL equivalent, low volume | Python logic needed at scale |

**Rule:** If a SQL UDF or built-in function can express the logic, use it. If you must use Python, use a **Pandas UDF** over a plain Python UDF for any significant data volume.

## Pandas UDFs — Vectorized

Pandas UDFs batch rows into a `pandas.Series` (or `DataFrame`), pass the whole batch to Python over Apache Arrow, and return a `pandas.Series`. The serialization cost is paid once per batch, not once per row — **orders of magnitude faster** than plain Python UDFs.

### Series to Series (Scalar)

One input column → one output column. The most common pattern.

```python
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StringType

@pandas_udf(StringType())
def mask_email_v(emails: pd.Series) -> pd.Series:
    def _mask(email):
        if pd.isna(email): return None
        local, domain = email.split('@')
        return local[:2] + '*' * (len(local) - 2) + '@' + domain
    return emails.apply(_mask)
```

### Series to Scalar (Aggregate)

Used as an aggregation function — takes a Series, returns a scalar.

```python
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import DoubleType

@pandas_udf(DoubleType())
def weighted_avg(values: pd.Series, weights: pd.Series) -> float:
    return (values * weights).sum() / weights.sum()

# Use as an aggregation
df.groupBy("region").agg(weighted_avg("amount", "weight"))
```

### Iterator of Series (for Expensive Initialization)

When the UDF needs expensive setup (loading an ML model, opening a DB connection), use the Iterator pattern — setup runs once per partition, not once per batch:

```python
from typing import Iterator

@pandas_udf(StringType())
def predict(iterator: Iterator[pd.Series]) -> Iterator[pd.Series]:
    model = load_model()    # runs ONCE per partition
    for batch in iterator:
        yield pd.Series(model.predict(batch))
```

In [ ]:
from pyspark.sql.functions import udf, pandas_udf, col
from pyspark.sql.types import StringType, DoubleType
import pandas as pd

# Plain Python UDF
@udf(returnType=StringType())
def mask_email(email):
    if email is None:
        return None
    parts = email.split('@')
    if len(parts) != 2:
        return email
    local, domain = parts
    return local[:2] + '*' * max(0, len(local) - 2) + '@' + domain

# Pandas UDF — same logic, vectorized
@pandas_udf(StringType())
def mask_email_v(emails: pd.Series) -> pd.Series:
    def _mask(email):
        if pd.isna(email):
            return None
        parts = email.split('@')
        if len(parts) != 2:
            return email
        local, domain = parts
        return local[:2] + '*' * max(0, len(local) - 2) + '@' + domain
    return emails.apply(_mask)

# Register Python UDF for SQL use
spark.udf.register("mask_email_sql", mask_email, StringType())

In [ ]:
emails_df = spark.createDataFrame(
    [("alice@example.com",), ("bob@databricks.com",), (None,), ("x@y.io",)],
    ["email"]
)

# Python UDF via PySpark
emails_df.withColumn("masked_py",     mask_email(col("email"))) \
         .withColumn("masked_pandas", mask_email_v(col("email"))) \
         .show(truncate=False)

In [ ]:
# Python UDF via spark.sql — uses the registered name
emails_df.createOrReplaceTempView("emails")
spark.sql("SELECT email, mask_email_sql(email) AS masked FROM emails").show(truncate=False)

In [ ]:
# Pandas UDF as an aggregation — custom percentile
@pandas_udf(DoubleType())
def p90(amounts: pd.Series) -> float:
    return float(amounts.quantile(0.90))

spark.createDataFrame([
    ("UK", 120.0), ("UK", 340.0), ("UK", 89.0), ("UK", 500.0), ("UK", 210.0),
    ("US", 50.0),  ("US", 400.0), ("US", 175.0)
], ["region", "amount"]) \
    .groupBy("region") \
    .agg(p90(col("amount")).alias("p90_amount")) \
    .show()

## Notebook Control Flow — dbutils

Databricks notebooks are not just interactive tools — they can be orchestrated programmatically as pipeline components using `dbutils`.

### Widgets — Parameterizing a Notebook

Widgets let you pass parameters into a notebook at runtime — from the Databricks Jobs UI, from another notebook, or from the API.

```python
# Declare widgets (usually at the top of the notebook)
dbutils.widgets.text("env",        "dev",        "Environment")       # free text
dbutils.widgets.text("start_date", "2024-01-01", "Start Date")
dbutils.widgets.dropdown("region", "UK", ["UK", "US", "DE", "FR"], "Region")

# Read widget values
env        = dbutils.widgets.get("env")
start_date = dbutils.widgets.get("start_date")
region     = dbutils.widgets.get("region")
```

**Using widget values in SQL cells:**

```sql
-- In a SQL cell, reference with ${widget_name}
SELECT * FROM silver.orders
WHERE  order_date >= '${start_date}'
  AND  region     =  '${region}'
```

**In a Python `spark.sql()` call — use an f-string:**

```python
spark.sql(f"""
  SELECT * FROM silver.orders
  WHERE order_date >= '{start_date}' AND region = '{region}'
""")
```

### Remove Widgets

```python
dbutils.widgets.remove("env")    # remove one
dbutils.widgets.removeAll()      # remove all
```

In [ ]:
# Declare parameters — when run as a Job, these are populated from the job config
dbutils.widgets.text("env",        "dev",        "Environment")
dbutils.widgets.text("start_date", "2024-01-01", "Start Date")
dbutils.widgets.dropdown("region", "UK", ["UK", "US", "DE", "FR"], "Region")

env        = dbutils.widgets.get("env")
start_date = dbutils.widgets.get("start_date")
region     = dbutils.widgets.get("region")

print(f"env={env}  start_date={start_date}  region={region}")

In [ ]:
# Use widget values inside spark.sql — f-string interpolation
result = spark.sql(f"""
  SELECT '{env}'        AS environment,
         '{start_date}' AS start_date,
         '{region}'     AS region,
         CURRENT_TIMESTAMP() AS run_ts
""")
result.show(truncate=False)

In [ ]:
# Conditional logic using widget values — branch pipeline behaviour
if env == "prod":
    target_table = "prod_catalog.silver.orders"
    write_mode   = "append"
else:
    target_table = "dev_catalog.silver.orders"
    write_mode   = "overwrite"

print(f"Writing to: {target_table}  mode: {write_mode}")

## Chaining Notebooks — `dbutils.notebook`

One notebook can call another as a sub-task and receive its return value. This enables **modular pipelines** built from individual notebook components.

### `dbutils.notebook.run` — Call a Notebook

```python
result = dbutils.notebook.run(
    path      = "/Repos/my-project/pipelines/silver_transform",
    timeout_seconds = 3600,
    arguments = {
        "env":        "prod",
        "start_date": "2024-01-15",
        "region":     "UK"
    }
)
print(f"Child notebook returned: {result}")
```

- The child notebook receives `arguments` as widget values — accessible with `dbutils.widgets.get()`
- The call is **synchronous** — the parent waits for the child to finish
- The return value is a string (whatever the child passes to `dbutils.notebook.exit()`)
- `timeout_seconds` caps the maximum wait time

### `dbutils.notebook.exit` — Return from a Notebook

```python
# At the end of the child notebook
rows_written = silver_df.count()
dbutils.notebook.exit(str(rows_written))   # must be a string
```

> **Exam tip:** The return value of `dbutils.notebook.run()` is always a **string**. If you need to pass structured data (e.g., a count or a status object), serialize it as JSON and parse it in the parent.

### When to Use Notebook Chaining vs Jobs

| | Notebook chaining (`notebook.run`) | Databricks Jobs (multi-task) |
|---|---|---|
| Runs on | Same cluster as parent | Each task can have its own cluster |
| Parallelism | Sequential | Tasks can run in parallel |
| Observability | Limited (parent notebook output) | Full job run UI, alerts, history |
| Retry logic | Manual | Built-in per-task retry |
| Production use | Simple cases | Recommended for production |

In [ ]:
# Early exit pattern — stop a notebook if preconditions aren't met
# Commonly used at the start of a pipeline notebook

def check_source_table_exists(table_name: str) -> bool:
    return spark.catalog.tableExists(table_name)

source_table = f"silver.orders_{region.lower()}"

if not check_source_table_exists(source_table):
    print(f"Source table '{source_table}' not found — exiting")
    dbutils.notebook.exit(f"SKIPPED: {source_table} not found")

# If we reach here, the table exists and we can proceed
print(f"Source table '{source_table}' found — proceeding")

## Mixing Python and SQL — Key Patterns

Python and SQL cells coexist in a Databricks notebook. The bridge between them is `spark.sql()` for running SQL from Python, and `createOrReplaceTempView()` for making a DataFrame queryable from SQL.

```python
# Python variable → SQL query via f-string
schema = "prod" if env == "prod" else "dev"
result = spark.sql(f"SELECT COUNT(*) FROM {schema}.orders")

# DataFrame → SQL via temp view
transformed_df.createOrReplaceTempView("transformed")
spark.sql("""
  INSERT INTO silver.orders
  SELECT * FROM transformed WHERE amount > 0
""")

# SQL result → Python variable
row_count = spark.sql("SELECT COUNT(*) FROM silver.orders").collect()[0][0]
print(f"Wrote {row_count} rows")
```

> **Security note:** When building SQL strings with f-strings, never interpolate raw user input — only interpolate trusted values like widget-provided env names or pre-validated dates. Interpolating arbitrary strings opens up SQL injection.

In [ ]:
# Full pattern: widget → Python logic → SQL write → exit with status

# 1. Read params
run_env    = dbutils.widgets.get("env")
run_region = dbutils.widgets.get("region")

# 2. Python logic — decide target based on env
target_schema = "prod_demo" if run_env == "prod" else "dev_demo"

# 3. Create the schema if needed
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema}")

# 4. Build a DataFrame using the region parameter
summary_df = spark.sql(f"""
  SELECT '{run_region}' AS region,
         CURRENT_DATE() AS run_date,
         42             AS rows_processed
""")

# 5. Make available in SQL, then write
summary_df.createOrReplaceTempView("run_summary")
spark.sql(f"""
  CREATE OR REPLACE TABLE {target_schema}.run_log
  USING DELTA AS SELECT * FROM run_summary
""")

# 6. Confirm and return
rows = spark.sql(f"SELECT COUNT(*) FROM {target_schema}.run_log").collect()[0][0]
print(f"Done. Wrote {rows} row(s) to {target_schema}.run_log")

# 7. Clean up demo
spark.sql(f"DROP SCHEMA IF EXISTS {target_schema} CASCADE")

In [ ]:
# Clean up widgets
dbutils.widgets.removeAll()

## Summary

- **Python UDFs** are created with `@udf(returnType=...)` or `udf(fn, returnType)`. Register with `spark.udf.register(name, fn, returnType)` to use them in SQL strings. Registration is session-scoped — not persisted.
- Python UDFs are slow at scale because they serialize data **per row** (pickle) across the JVM-Python boundary. Prefer SQL UDFs or built-ins first; use Python UDFs only when necessary.
- **Pandas UDFs** (`@pandas_udf`) batch rows via Apache Arrow — much faster than row-at-a-time Python UDFs. Three patterns: Series→Series (scalar transform), Series→Scalar (aggregate), Iterator of Series (expensive init per partition, e.g., loading an ML model).
- **`dbutils.widgets`** parameterizes a notebook: `text`, `dropdown`, `combobox`. Read values with `dbutils.widgets.get(name)`. Reference in SQL cells with `${name}`; in Python `spark.sql()` use f-string interpolation.
- **`dbutils.notebook.run(path, timeout, arguments)`** calls a child notebook synchronously and returns its exit string. **`dbutils.notebook.exit(value)`** returns a string to the caller.
- For production pipelines, prefer **Databricks Jobs** (multi-task) over notebook chaining — Jobs provide per-task clusters, parallelism, retries, and full observability.
- The Python ↔ SQL bridge: `spark.sql(f"...")` runs SQL from Python; `df.createOrReplaceTempView(name)` makes a DataFrame queryable from SQL; `.collect()[0][0]` extracts a scalar result back to Python.

---

**Section 2 — ELT with Spark SQL & Python — complete.** The next section covers incremental data processing: Auto Loader, CDC with MERGE, and Structured Streaming on Databricks.

Next: `09-auto-loader.ipynb`